In [2]:
import sys
import pandas as pd
from pathlib import Path
print(sys.executable)
print(pd.__version__)

c:\Users\caswv\projects\Investment-Portfolio-Analysis\.venv\Scripts\python.exe
3.0.5


In [3]:
DATA_PATH = Path("../data/raw/portfolio_20261031.xlsx")

print(DATA_PATH.resolve())
print(DATA_PATH.exists())

C:\Users\caswv\projects\Investment-Portfolio-Analysis\data\raw\portfolio_20261031.xlsx
True


In [ ]:
quality_raw =   pd.read_excel(DATA_PATH, sheet_name="1) Kwaliteit", header=None)
quality_raw.shape

(102, 27)

In [5]:
toy = pd.DataFrame({
    0: ["Titel", None, "Ticker", "V", "MCO"],
    1: [None, None, "Naam", "Visa", "Moody's"],
})
toy

,0,1
0,Titel,NaN
1,NaN,NaN
2,Ticker,Naam
3,V,Visa
4,MCO,Moody's


In [13]:
header_mask = quality_raw[0] == "Ticker"
header_mask

0      False
1      False
2      False
3      False
4      False
       ...  
97     False
98     False
99     False
100    False
101    False
Name: 0, Length: 102, dtype: bool

In [14]:
quality_raw[header_mask]

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
26,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.,Land,Beurs,...,CHF,CAD,Overig ontw.,Opkomend,Controle valuta,Hoofdvaluta,Div. rendement,Cijfer,Rating,Rol: waarom deze positie


In [15]:
quality_raw.index[header_mask]

RangeIndex(start=26, stop=27, step=1)

In [16]:
header_row = quality_raw.index[header_mask][0]
header_row

26

In [17]:
quality = pd.read_excel(DATA_PATH, sheet_name="1) Kwaliteit", header=header_row)
quality.shape

(75, 27)

In [18]:
quality.iloc[14:25, :8]

,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.
14,EW,Edwards Lifesciences,US28176E1082,Aandeel,Medtech,Kwaliteit,0.05,0.0135
15,SYK,Stryker,US8636671013,Aandeel,Medtech,Kwaliteit,0.05,0.0135
16,STMN,Straumann,CH1175448666,Aandeel,Medtech,Kwaliteit,0.05,0.0135
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,Totaal,NaN,NaN,NaN,NaN,NaN,1.0,0.27
21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,Themaverdeling,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,Thema (sleutel),Omschrijving,Weging pie,Weging portf.,Aantal,NaN,NaN,NaN


In [19]:
empty_ticker_mask = quality["Ticker"].isna()
empty_ticker_mask.head(20)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17     True
18     True
19     True
Name: Ticker, dtype: bool

In [20]:
first_empty = quality.index[empty_ticker_mask][0]
first_empty

np.int64(17)

In [21]:
quality_holdings = quality.iloc[:first_empty]
quality_holdings.shape

(17, 27)

In [22]:
quality_holdings.tail(3)

,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.,Land,Beurs,...,CHF,CAD,Overig ontw.,Opkomend,Controle valuta,Hoofdvaluta,Div. rendement,Cijfer,Rating,Rol: waarom deze positie
14,EW,Edwards Lifesciences,US28176E1082,Aandeel,Medtech,Kwaliteit,0.05,0.0135,VS,Xetra,...,0.0,0.0,0.0,0.0,OK,USD,0.0000,8.4,Buy,Leider in TAVR; ander verdienmodel dan ISRG en...
15,SYK,Stryker,US8636671013,Aandeel,Medtech,Kwaliteit,0.05,0.0135,VS,NYSE,...,0.0,0.0,0.0,0.0,OK,USD,0.0121,8.1,Buy,Orthopedie en medsurg tegen een historisch lag...
16,STMN,Straumann,CH1175448666,Aandeel,Medtech,Kwaliteit,0.05,0.0135,Zwitserland,SIX,...,1.0,0.0,0.0,0.0,OK,CHF,0.0103,8.1,Buy,Leider in premium tandimplantaten; CHF; gekoze...


In [23]:
old_weights = pd.to_numeric(quality["Weging pie"], errors="coerce")
old_filter = old_weights.notna()

print(old_filter.sum())
print(old_weights[old_filter].sum())

36
132.7


In [24]:
quality_holdings["Weging pie"].sum()

1.0000000000000004

In [25]:
def pie_weight_sum(holdings):
    """Return the sum of the pie weights of a holdings table."""
    return holdings["Weging pie"].sum()


pie_weight_sum(quality_holdings)

1.0000000000000004

In [28]:
def read_holdings(path, sheet_name):
    """Read the holdings table from one pie sheet.

    Finds the header row (first column equals "Ticker"), re-reads the sheet
    from that row and cuts the table off at the first empty ticker.
    """
    # 1. Read the sheet without a header and find the header row
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_mask = raw[0] == "Ticker"
    header_row = raw[header_mask].index[0]

    # 2. Re-read the sheet with the correct header row
    table = pd.read_excel(path, sheet_name=sheet_name, header=header_row)

    # 3. Cut the table off at the first empty ticker
    empty_ticker_mask = table["Ticker"].isna()
    first_empty = table.index[empty_ticker_mask][0]
    return table.iloc[:first_empty]

In [30]:
read_holdings(DATA_PATH, "1) Kwaliteit").shape

(17, 27)

In [31]:
read_holdings(DATA_PATH, "3) AI Boom").shape

(8, 27)

In [32]:
quality.iloc[:first_empty + 1].tail(2)

,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.,Land,Beurs,...,CHF,CAD,Overig ontw.,Opkomend,Controle valuta,Hoofdvaluta,Div. rendement,Cijfer,Rating,Rol: waarom deze positie
16,STMN,Straumann,CH1175448666,Aandeel,Medtech,Kwaliteit,0.05,0.0135,Zwitserland,SIX,...,1.0,0.0,0.0,0.0,OK,CHF,0.0103,8.1,Buy,Leider in premium tandimplantaten; CHF; gekoze...
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
numbers = [1, 2, 3]
squares = []
for n in numbers:
    squares.append(n ** 2)

squares

[1, 4, 9]

In [34]:
PIE_SHEETS = [
    "1) Kwaliteit",
    "2) Thematisch",
    "3) AI Boom",
    "4) Energie",
    "5) Mijnbouw",
    "6) Multipolaire Wereldorde",
]

In [35]:
frames = []

holdings = read_holdings(DATA_PATH, "1) Kwaliteit")
holdings["pie_sheet"] = "1) Kwaliteit"
frames.append(holdings)

len(frames)

1

In [36]:
holdings[["Ticker", "pie_sheet"]].head(3)

,Ticker,pie_sheet
0,V,1) Kwaliteit
1,MCO,1) Kwaliteit
2,LLY,1) Kwaliteit


In [64]:
frames = []
for sheet_name in PIE_SHEETS:
    holdings = read_holdings(DATA_PATH, sheet_name)
    holdings["pie_sheet"] = sheet_name
    holdings = holdings.rename(columns=CATEGORY_COLUMNS)
    frames.append(holdings)
len(frames)

6

In [65]:
all_holdings = pd.concat(frames, ignore_index=True)
all_holdings.shape

(55, 28)

In [66]:
all_holdings.columns

Index(['Ticker', 'Naam', 'ISIN', 'Type', 'category', 'Model', 'Weging pie',
       'Weging portf.', 'Land', 'Beurs', 'Not. valuta', 'TER', 'Replicatie',
       'Uitkering', 'USD', 'EUR', 'GBP', 'CHF', 'CAD', 'Overig ontw.',
       'Opkomend', 'Controle valuta', 'Hoofdvaluta', 'Div. rendement',
       'Cijfer', 'Rating', 'Rol: waarom deze positie', 'pie_sheet'],
      dtype='str')

In [67]:
holdings[["Ticker", "pie_sheet"]].head(3)

,Ticker,pie_sheet
0,EPIN,6) Multipolaire Wereldorde
1,AMEL,6) Multipolaire Wereldorde
2,CHN,6) Multipolaire Wereldorde


In [68]:
all_holdings = pd.concat(frames, ignore_index=True)
all_holdings.shape

(55, 28)

In [69]:
all_holdings["pie_sheet"].value_counts()

pie_sheet
1) Kwaliteit                  17
4) Energie                    10
5) Mijnbouw                    9
3) AI Boom                     8
6) Multipolaire Wereldorde     6
2) Thematisch                  5
Name: count, dtype: int64

In [70]:
CATEGORY_COLUMNS = {"Thema": "category", "Ketenlaag": "category", "Regio": "category"}

In [73]:
COLUMN_NAMES = {
    "Ticker": "ticker",
    "Naam": "name",
    "ISIN": "isin",
    "Type": "asset_type",
    "Model": "model",
    "Weging pie": "weight_in_pie",
    "Weging portf.": "weight_in_portfolio",
    "Land": "country",
    "Beurs": "exchange",
    "Not. valuta": "listing_currency",
    "TER": "ter",
    "Replicatie": "replication",
    "Uitkering": "distribution",
    "USD": "econ_usd",
    "EUR": "econ_eur",
    "GBP": "econ_gbp",
    "CHF": "econ_chf",
    "CAD": "econ_cad",
    "Overig ontw.": "econ_other_dm",
    "Opkomend": "econ_em",
    "Controle valuta": "econ_check",
    "Hoofdvaluta": "main_currency",
    "Div. rendement": "dividend_yield",
    "Cijfer": "score",
    "Rating": "rating",
    "Rol: waarom deze positie": "role",
}
all_holdings = all_holdings.rename(columns=COLUMN_NAMES)
list(all_holdings.columns)

['ticker',
 'name',
 'isin',
 'asset_type',
 'category',
 'model',
 'weight_in_pie',
 'weight_in_portfolio',
 'country',
 'exchange',
 'listing_currency',
 'ter',
 'replication',
 'distribution',
 'econ_usd',
 'econ_eur',
 'econ_gbp',
 'econ_chf',
 'econ_cad',
 'econ_other_dm',
 'econ_em',
 'econ_check',
 'main_currency',
 'dividend_yield',
 'score',
 'rating',
 'role',
 'pie_sheet']